In [1]:
import os

import torch
import numpy as np
import pandas as pd
from transformers import pipeline
from supabase import create_client, Client
from dotenv import load_dotenv

from models import ModernBERT_v2, ModelInput

load_dotenv()
torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(supabase_url=url, supabase_key=key)

In [2]:
# Get chunk data
chunk_df = pd.read_csv(
    "/home/jovyan/active-projects/shared-resources/strapi-exports/data/strapi_data.csv",
    index_col=0,
)
# There are some duplicate chunks. Likely multiple pages linked to the same chunk.
chunk_df = chunk_df.loc[~chunk_df.index.duplicated(keep="first"), :]
chunk_df.index.value_counts()

chunk_slug
Discussion-II-5852                                       1
Structure-and-Function-of-the-Respiratory-System-6247    1
The-Conducting-Airways-6248                              1
The-Respiratory-Airways-6249                             1
Gas-Exchange-6250                                        1
                                                        ..
16-Training-Records-2031                                 1
17-Training-Program-Evaluation-2032                      1
31-Job-Training-and-Transfer-Training-2023               1
33-Refresher-Training-2024                               1
41-Requirements-and-Placement-2039                       1
Name: count, Length: 585, dtype: int64

In [3]:
# Get logs data
client_names = ["hinze_rmp_summer_2025", "middle_georgia"]

logs_response = (
    supabase.table("logs")
    .select("*")
    # .in_("client_name", client_names)
    .eq("api_endpoint", "/score/answer")
    .order("created_at", desc=True)
    .limit(300)
    .execute()
)

logs_df = pd.json_normalize(logs_response.data)

# Join
df = logs_df.merge(
    chunk_df, left_on="request_body.chunk_slug", right_on="chunk_slug", how="left"
)

# Construct DataFrame
columns_to_keep = [
    "volume_slug",
    "request_body.page_slug",
    "request_body.chunk_slug",
    "chunk_text",
    "question",
    "answer",
    "request_body.answer",
    "response_body.level",
    "response_body.score",
    "response_body.feedback",
    "response_body.is_passing",
]

df = df[columns_to_keep]
df = df.rename(
    columns={
        "answer": "reference",
        "request_body.answer": "candidate",
        "chunk_text": "text",
    }
)

# Clean up column names
df.columns = [
    col.replace("request_body.", "").replace("response_body.", "") for col in df.columns
]

df

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing
0,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,State-Governments-4457,The fourth section of the Constitution explain...,What is the person in charge of the executive ...,A governor.,A governor,2,2.212706,"You are on the right track, but your response ...",False
1,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,State-Governments-4457,The fourth section of the Constitution explain...,What is the person in charge of the executive ...,A governor.,A Governor,2,1.947339,"You are on the right track, but your response ...",False
2,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,State-Governments-4457,The fourth section of the Constitution explain...,What is the person in charge of the executive ...,A governor.,Governor,1,1.613354,Your response is missing many relevant details...,False
3,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,The-US-Constitution-4445,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.,It was made in 1787,3,2.271365,Good job. You included some relevant details f...,True
4,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,The-US-Constitution-4445,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.,I don't know!,1,1.339946,Your response is missing many relevant details...,False
...,...,...,...,...,...,...,...,...,...,...,...
295,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,N-gram-strength-of-association-4819,N-gram strength of association\n\nStrength-of-...,What do strength-of-association norms measure?,Strength-of-association norms measure the cond...,Strength-of-association norms measure the cond...,3,2.998177,Good job. You included some relevant details f...,True
296,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,N-gram-strength-of-association-4819,N-gram strength of association\n\nStrength-of-...,What do strength-of-association norms measure?,Strength-of-association norms measure the cond...,They measure the conditional probability that ...,2,1.797399,"You are on the right track, but your response ...",False
297,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,N-gram-strength-of-association-4819,N-gram strength of association\n\nStrength-of-...,What do strength-of-association norms measure?,Strength-of-association norms measure the cond...,They measure the likelihood that words occur t...,2,1.899061,"You are on the right track, but your response ...",False
298,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,Corpus-driven-approaches-4816,Other approaches to measuring contextual disti...,What are some approaches to measuring contextu...,One approach takes a lexical perspective and m...,They measure how often a word co-occurs with o...,3,2.905182,Good job. You included some relevant details f...,True


In [4]:
mb_w_reference_answer_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_authentic_multirc_with_reference_with_label"

MBV2 = ModernBERT_v2(mb_w_reference_answer_path)

Device set to use cuda


In [5]:
def score(df, pipe):
    all_preds = []

    for ex in df.to_dict("records"):
        model_input = ModelInput.from_dict(ex)
        pred = pipe(model_input)
        all_preds.append(pred)

    return all_preds


df["new_score"] = score(df, MBV2)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [11]:
(df["score"] > 2) & (df["new_score"] < 2)

0      False
1      False
2      False
3      False
4      False
       ...  
295    False
296    False
297    False
298    False
299    False
Length: 300, dtype: bool

In [12]:
filters = (
    (abs(df["score"] - df["new_score"]) > 1.0)  # scores differ by more than 1.0
    | (
        (df["score"] > 2) & (df["new_score"] < 2)
    )  # scores on different sides of threshold
    | (
        (df["score"] < 2) & (df["new_score"] > 2)
    )  # scores on different sides of threshold
)

df_filtered = df[filters]
df_filtered

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing,new_score
1,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,State-Governments-4457,The fourth section of the Constitution explain...,What is the person in charge of the executive ...,A governor.,A Governor,2,1.947339,"You are on the right track, but your response ...",False,2.261888
2,one-nation-one-people-the-uscis-civics-test-te...,the-u-s-constitution,State-Governments-4457,The fourth section of the Constitution explain...,What is the person in charge of the executive ...,A governor.,Governor,1,1.613354,Your response is missing many relevant details...,False,2.045139
58,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,They're revising TOEFL by using T2K-SWAL corpu...,1,1.514471,Your response is missing many relevant details...,False,2.604988
65,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,by checking the consistency of the test langua...,2,1.689498,"You are on the right track, but your response ...",False,2.607835
66,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,o check the consistency of the test language w...,1,1.617471,Your response is missing many relevant details...,False,2.508170
67,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,By using data from the T2K-SWAL corpus to ensu...,2,1.795371,"You are on the right track, but your response ...",False,2.898842
68,natural-language-processing,speaking-and-writing-in-the-university-a-multi...,Implications-for-Materials-Development-5239,This study has powerful implications for test ...,What is Educational Testing Service currently ...,Educational Testing Service is revising the TO...,By using data from the T2K-SWAL corpus to ensu...,1,1.347509,Your response is missing many relevant details...,False,2.027951
78,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Transformation-Based-Error-Driven-Learning-5094,Transformation-based error-driven learning has...,What are the three things that must be specifi...,"The initial state annotator, the space of tran...","speech tagging, prepositional phrase attachmen...",2,1.833597,"You are on the right track, but your response ...",False,2.463137
103,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,Discussion-4857,This study introduces and helps validate TAALE...,What is the purpose of TAALES 2.0 as described...,TAALES 2.0 is a tool for measuring lexical sop...,It is an easy to use freely available versatil...,2,1.906836,"You are on the right track, but your response ...",False,2.655530
108,natural-language-processing,sentiment-analysis-and-social-cognition-engine...,Discussion-5771,"This article introduces a new tool, SEANCE, wh...",How did SEANCE perform compared to LIWC in sen...,SEANCE's individual indices and component scor...,The pairwise comparisons indicated that the SE...,1,1.583999,Your response is missing many relevant details...,False,2.222171


In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Assuming your dataframe is called 'df'
current_idx = 0

# Configuration
PASSING_THRESHOLD = 2.0  # Scores >= this value are passing


def create_display(df, idx):
    """Create the display for a given row index"""
    if idx < 0 or idx >= len(df):
        return None

    row = df.iloc[idx]

    # Round scores to 2 decimal places
    score = round(row["score"], 2)
    new_score = round(row["new_score"], 2)

    # Determine passing status and styling
    score_passing = score >= PASSING_THRESHOLD
    new_score_passing = new_score >= PASSING_THRESHOLD

    score_bg = "#c8e6c9" if score_passing else "#ffcdd2"
    new_score_bg = "#c8e6c9" if new_score_passing else "#ffcdd2"

    html_content = f"""
    <div style="font-family: Arial, sans-serif; line-height: 1.6;">
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin-bottom: 15px;">
            <strong>Row {idx + 1} of {len(df)}</strong>
        </div>
        
        <div style="background-color: #fff3e0; padding: 12px; border-left: 4px solid #FF9800; margin-bottom: 15px;">
            <strong>Text/Context:</strong><br>
            {row["text"]}
        </div>
        
        <div style="background-color: #e8f4f8; padding: 12px; border-left: 4px solid #2196F3; margin-bottom: 15px;">
            <strong>Question:</strong><br>
            {row["question"]}
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <div style="background-color: {score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">Original Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{score}</div>
            </div>
            
            <div style="background-color: {new_score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">New Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{new_score}</div>
            </div>
        </div>
        
        <hr style="margin: 20px 0;">
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="background-color: #e8f5e9; padding: 15px; border: 2px solid #4CAF50; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #2e7d32;">Reference Answer</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["reference"]}
                </div>
            </div>
            
            <div style="background-color: #fce4ec; padding: 15px; border: 2px solid #E91E63; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #880e4f;">Student Response</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["candidate"]}
                </div>
            </div>
        </div>
    </div>
    """

    return html_content


# Create output widget
output = widgets.Output()

# Create buttons
prev_button = widgets.Button(description="← Previous", button_style="info")
next_button = widgets.Button(description="Next →", button_style="info")
status_label = widgets.Label(value=f"Row 1 of {len(df_filtered)}")


def on_prev_clicked(b):
    global current_idx
    current_idx = max(0, current_idx - 1)
    update_display()


def on_next_clicked(b):
    global current_idx
    current_idx = min(len(df_filtered) - 1, current_idx + 1)
    update_display()


def update_display():
    output.clear_output(wait=True)
    html_content = create_display(df_filtered, current_idx)
    status_label.value = f"Row {current_idx + 1} of {len(df_filtered)}"
    with output:
        display(HTML(html_content))


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Create layout
button_box = widgets.HBox([prev_button, status_label, next_button])
layout = widgets.VBox([output, button_box])

# Initial display
update_display()

# Show the interface
display(layout)